# Batch Board Warping from UNet Segmentation (Production Pipeline)

**Goal:** Warp all chessboard images using UNet-based segmentation masks.

**Pipeline:**
1. Load trained UNet model (`best_unet_model.pth`)
2. Run batch inference on all images in `Board_coordinates/{train,validation,test}/images/`
3. Extract 4 corner coordinates from masks robustly
4. Warp to 800x800 top-down view
5. Save warped images, homography matrices (M, Minv), and debug visualizations

**Note:** This is the PRODUCTION pipeline. For training, see `03_unet_training.ipynb`. For single-image demo, see `03b_unet_inference_demo.ipynb`.

In [ ]:
# Install required packages if not available
import subprocess
import sys

def install_package(package):
    """Install a package if not already installed."""
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

# Install PyTorch and segmentation-models-pytorch if needed
try:
    import torch
    import torch.nn as nn
    import segmentation_models_pytorch as smp
    print("✓ PyTorch and segmentation-models-pytorch are available")
except ImportError:
    print("Installing required packages...")
    install_package("torch")
    install_package("torchvision")
    install_package("segmentation-models-pytorch")
    import torch
    import torch.nn as nn
    import segmentation_models_pytorch as smp
    print("✓ Installation complete")

# Standard imports
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set up paths
possible_roots = [
    Path.cwd().parent,  # If running from notebooks/ directory
    Path.cwd(),         # If running from workspace root
]

PROJECT_ROOT = None
for root in possible_roots:
    test_path = root / "data" / "raw" / "Chess Pieces Detection Image Dataset" / "Board_coordinates"
    if test_path.exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()

# Paths
BOARD_COORDS_BASE = PROJECT_ROOT / "data" / "raw" / "Chess Pieces Detection Image Dataset" / "Board_coordinates"
UNET_MODEL_PATH = PROJECT_ROOT / "notebooks" / "best_unet_model.pth"
OUTPUT_BASE = PROJECT_ROOT / "data" / "processed" / "warped_unet"

# Constants
IMAGE_SIZE = 320  # UNet training size
WARP_SIZE = 800   # Output warped size (matches notebook 04)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Project root: {PROJECT_ROOT}")
print(f"UNet model path: {UNET_MODEL_PATH}")
print(f"Model exists: {UNET_MODEL_PATH.exists()}")
print(f"Output base: {OUTPUT_BASE}")
print(f"Device: {DEVICE}")
print(f"Warp size: {WARP_SIZE}x{WARP_SIZE}")

✓ PyTorch and segmentation-models-pytorch are available
Project root: /Users/ibrahimcbc/Library/CloudStorage/GoogleDrive-icebecioglu21@ku.edu.tr/My Drive/COMP411/assignments/COMP-411_Project/chessvision
UNet model path: /Users/ibrahimcbc/Library/CloudStorage/GoogleDrive-icebecioglu21@ku.edu.tr/My Drive/COMP411/assignments/COMP-411_Project/chessvision/notebooks/best_unet_model.pth
Model exists: True
Output base: /Users/ibrahimcbc/Library/CloudStorage/GoogleDrive-icebecioglu21@ku.edu.tr/My Drive/COMP411/assignments/COMP-411_Project/chessvision/data/processed/warped_unet
Device: cpu
Warp size: 800x800


## A) Load UNet Model

In [ ]:
def load_unet_model(model_path):
    """Load trained UNet model with exact architecture from notebook 03."""
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,  # We're loading our own weights
        in_channels=3,
        classes=1,
        activation=None
    )
    model.load_state_dict(torch.load(str(model_path), map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return model

# Load model
if UNET_MODEL_PATH.exists():
    unet_model = load_unet_model(UNET_MODEL_PATH)
    preprocessing_fn = smp.encoders.get_preprocessing_fn('resnet34', 'imagenet')
    print("✓ UNet model loaded successfully")
else:
    raise FileNotFoundError(f"Model not found: {UNET_MODEL_PATH}")

✓ UNet model loaded successfully


## B) Helper Functions: Inference, Corner Extraction, Validation

In [ ]:
def predict_mask(model, image_rgb, preprocessing_fn):
    """Run UNet inference on an image and return binary mask at original resolution."""
    original_h, original_w = image_rgb.shape[:2]

    # Resize to training size
    img_resized = cv2.resize(image_rgb, (IMAGE_SIZE, IMAGE_SIZE))

    # Preprocess (normalize)
    img_input = preprocessing_fn(img_resized)

    # To Tensor (C, H, W)
    img_tensor = torch.from_numpy(img_input.transpose(2, 0, 1)).float().unsqueeze(0)
    img_tensor = img_tensor.to(DEVICE)

    # Inference
    with torch.no_grad():
        output = model(img_tensor)
        # Apply sigmoid since we trained with logits (activation=None)
        prob_mask = torch.sigmoid(output).squeeze().cpu().numpy()

    # Threshold and resize back to original size
    binary_mask = (prob_mask > 0.5).astype(np.uint8) * 255
    full_size_mask = cv2.resize(binary_mask, (original_w, original_h), interpolation=cv2.INTER_NEAREST)

    return full_size_mask

def order_corners(pts):
    """Order 4 corners as [TL, TR, BR, BL]."""
    rect = np.zeros((4, 2), dtype="float32")
    # Top-left: smallest sum
    # Bottom-right: largest sum
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]  # TL
    rect[2] = pts[np.argmax(s)]  # BR
    # Top-right: smallest diff
    # Bottom-left: largest diff
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]  # TR
    rect[3] = pts[np.argmax(diff)]  # BL
    return rect

def extract_corners_from_mask(mask):
    """
    Extract 4 corners from binary mask robustly.
    Returns ordered corners [TL, TR, BR, BL] or None if failed.
    """
    # 1. Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None

    # 2. Get largest contour
    c = max(contours, key=cv2.contourArea)

    # 3. Try approxPolyDP first
    peri = cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, 0.02 * peri, True)

    if len(approx) == 4:
        pts = approx.reshape(4, 2)
    else:
        # Fallback 1: minAreaRect
        rect = cv2.minAreaRect(c)
        box = cv2.boxPoints(rect)
        pts = box

        # If still not 4 points after reshape, try convex hull
        if len(pts) != 4:
            hull = cv2.convexHull(c)
            if len(hull) >= 4:
                # Try to approximate hull to 4 points
                peri = cv2.arcLength(hull, True)
                approx = cv2.approxPolyDP(hull, 0.1 * peri, True)
                if len(approx) >= 4:
                    # Take first 4 points
                    pts = approx[:4].reshape(4, 2)
                else:
                    # Last resort: bounding rect
                    x, y, w, h = cv2.boundingRect(c)
                    pts = np.array([
                        [x, y],
                        [x + w, y],
                        [x + w, y + h],
                        [x, y + h]
                    ], dtype=np.float32)
            else:
                return None

    # Order corners
    ordered_corners = order_corners(pts)
    return ordered_corners

def validate_corners(corners, img_width, img_height, min_area_ratio=0.05, min_pairwise_dist=2.0):
    """Validate corner coordinates."""
    if corners is None:
        return False, "No corners provided"

    if corners.shape != (4, 2):
        return False, f"Invalid shape: {corners.shape}, expected (4, 2)"

    # Check bounds
    if np.any(corners < 0) or np.any(corners[:, 0] >= img_width) or np.any(corners[:, 1] >= img_height):
        return False, f"Corners out of bounds (image: {img_width}x{img_height})"

    # Check uniqueness (minimum pairwise distance)
    pairwise_dists = []
    for i in range(4):
        for j in range(i + 1, 4):
            dist = np.linalg.norm(corners[i] - corners[j])
            pairwise_dists.append(dist)

    min_dist = min(pairwise_dists) if pairwise_dists else 0.0
    if min_dist < min_pairwise_dist:
        return False, f"Duplicate corners: min_dist={min_dist:.2f} < {min_pairwise_dist}"

    # Check area
    area = cv2.contourArea(corners)
    img_area = img_width * img_height
    area_ratio = area / img_area

    if area_ratio < min_area_ratio:
        return False, f"Area too small: {area_ratio:.4f} < {min_area_ratio}"

    return True, area_ratio

print("✓ Helper functions defined")

✓ Helper functions defined


## C) Batch Processing Function

In [ ]:
def draw_chess_grid(image_rgb, grid_size=8, line_color_rgb=(255, 0, 0), line_thickness=2):
    """Draw 8x8 grid overlay on image."""
    img_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    img_with_grid = img_bgr.copy()
    h, w = image_rgb.shape[:2]

    line_color_bgr = (line_color_rgb[2], line_color_rgb[1], line_color_rgb[0])
    square_size_x = w / grid_size
    square_size_y = h / grid_size

    # Draw vertical lines
    for i in range(grid_size + 1):
        x = int(i * square_size_x)
        cv2.line(img_with_grid, (x, 0), (x, h), line_color_bgr, line_thickness)

    # Draw horizontal lines
    for i in range(grid_size + 1):
        y = int(i * square_size_y)
        cv2.line(img_with_grid, (0, y), (w, y), line_color_bgr, line_thickness)

    img_with_grid_rgb = cv2.cvtColor(img_with_grid, cv2.COLOR_BGR2RGB)
    return img_with_grid_rgb

def process_split(split_name, debug_n=12):
    """Process all images in a split: UNet inference → corners → warp → save."""
    print(f"\n{'='*60}")
    print(f"Processing {split_name} split")
    print(f"{'='*60}")

    # Paths
    images_dir = BOARD_COORDS_BASE / split_name / "images"
    output_images_dir = OUTPUT_BASE / split_name / "images"
    output_matrices_dir = OUTPUT_BASE / split_name / "matrices"
    output_debug_dir = OUTPUT_BASE / split_name / "debug"

    output_images_dir.mkdir(parents=True, exist_ok=True)
    output_matrices_dir.mkdir(parents=True, exist_ok=True)
    output_debug_dir.mkdir(parents=True, exist_ok=True)

    # Get all image files
    image_files = sorted(list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.png")))
    print(f"Found {len(image_files)} images")

    # Statistics
    stats = {
        'total_images': len(image_files),
        'warped_ok': 0,
        'skipped_no_mask': 0,
        'skipped_no_corners': 0,
        'skipped_invalid_corners': 0,
        'invalid_corner_reasons': defaultdict(int),
        'area_ratios': []
    }

    # Destination points for warping (TL, TR, BR, BL)
    dst_points = np.array([
        [0, 0],
        [WARP_SIZE, 0],
        [WARP_SIZE, WARP_SIZE],
        [0, WARP_SIZE]
    ], dtype=np.float32)

    for idx, image_path in enumerate(image_files):
        stem = image_path.stem

        # Load image
        img_bgr = cv2.imread(str(image_path))
        if img_bgr is None:
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]

        # A) UNet inference
        mask = predict_mask(unet_model, img_rgb, preprocessing_fn)

        if mask is None or np.sum(mask > 0) < 100:  # Minimum mask area
            stats['skipped_no_mask'] += 1
            continue

        # B) Extract corners from mask
        corners = extract_corners_from_mask(mask)

        if corners is None:
            stats['skipped_no_corners'] += 1
            continue

        # C) Validate corners
        is_valid, validation_result = validate_corners(corners, w, h, min_area_ratio=0.05, min_pairwise_dist=2.0)

        if not is_valid:
            stats['skipped_invalid_corners'] += 1
            reason = validation_result
            stats['invalid_corner_reasons'][reason] += 1
            continue

        area_ratio = validation_result
        stats['area_ratios'].append(area_ratio)

        # D) Warp
        M = cv2.getPerspectiveTransform(corners, dst_points)
        warped = cv2.warpPerspective(img_rgb, M, (WARP_SIZE, WARP_SIZE))

        # E) Save warped image
        warped_bgr = cv2.cvtColor(warped, cv2.COLOR_RGB2BGR)
        warped_path = output_images_dir / f"{stem}.png"
        cv2.imwrite(str(warped_path), warped_bgr)

        # F) Save homography matrices
        M_path = output_matrices_dir / f"{stem}_M.npy"
        Minv_path = output_matrices_dir / f"{stem}_Minv.npy"
        np.save(str(M_path), M)
        np.save(str(Minv_path), np.linalg.inv(M))

        stats['warped_ok'] += 1

        # G) Debug outputs for first N images
        if idx < debug_n:
            # Debug 1: Mask overlay with corners
            overlay = img_rgb.copy()
            mask_colored = cv2.applyColorMap(mask, cv2.COLORMAP_JET)
            overlay = cv2.addWeighted(overlay, 0.7, mask_colored, 0.3, 0)

            # Draw corners
            corners_int = corners.astype(np.int32)
            cv2.drawContours(overlay, [corners_int], -1, (0, 255, 0), 3)
            for i, (label, pt) in enumerate(zip(['TL', 'TR', 'BR', 'BL'], corners)):
                cv2.circle(overlay, (int(pt[0]), int(pt[1])), 8, (255, 0, 0), -1)
                cv2.putText(overlay, label, (int(pt[0]) + 10, int(pt[1])),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

            overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
            debug_mask_path = output_debug_dir / f"{stem}_mask_corners.png"
            cv2.imwrite(str(debug_mask_path), overlay_bgr)

            # Debug 2: Warped with grid
            warped_with_grid = draw_chess_grid(warped, grid_size=8)
            warped_grid_bgr = cv2.cvtColor(warped_with_grid, cv2.COLOR_RGB2BGR)
            debug_warped_path = output_debug_dir / f"{stem}_warped_grid.png"
            cv2.imwrite(str(debug_warped_path), warped_grid_bgr)

        if (idx + 1) % 10 == 0:
            print(f"  Processed {idx + 1}/{len(image_files)} images...")

    return stats

print("✓ Processing functions defined")

✓ Processing functions defined


## D) Process All Splits

In [ ]:
# Process all splits
all_stats = {}

for split_name in ['train', 'validation', 'test']:
    images_dir = BOARD_COORDS_BASE / split_name / "images"
    if images_dir.exists() and len(list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.png"))) > 0:
        stats = process_split(split_name, debug_n=12)
        all_stats[split_name] = stats
    else:
        print(f"\n No images found for {split_name} split, skipping...")


Processing train split
Found 129 images
  Processed 10/129 images...
  Processed 20/129 images...
  Processed 30/129 images...
  Processed 40/129 images...
  Processed 50/129 images...
  Processed 60/129 images...
  Processed 70/129 images...
  Processed 80/129 images...
  Processed 90/129 images...
  Processed 100/129 images...
  Processed 110/129 images...
  Processed 120/129 images...

Processing validation split
Found 42 images
  Processed 10/42 images...
  Processed 20/42 images...
  Processed 30/42 images...
  Processed 40/42 images...

Processing test split
Found 43 images
  Processed 10/43 images...
  Processed 20/43 images...
  Processed 30/43 images...
  Processed 40/43 images...


## E) Summary Statistics

In [ ]:
print("\n" + "="*80)
print("BATCH WARPING SUMMARY (UNet-based)")
print("="*80)

for split_name, stats in all_stats.items():
    print(f"\n{split_name.upper()} SPLIT:")
    print(f"  Total images:           {stats['total_images']}")
    print(f"  Warped successfully:    {stats['warped_ok']}")
    print(f"  Skipped (no mask):      {stats['skipped_no_mask']}")
    print(f"  Skipped (no corners):   {stats['skipped_no_corners']}")
    print(f"  Skipped (invalid):      {stats['skipped_invalid_corners']}")

    if stats['warped_ok'] > 0:
        success_rate = 100.0 * stats['warped_ok'] / stats['total_images']
        print(f"  Success rate:           {success_rate:.1f}%")

        if stats['area_ratios']:
            area_mean = np.mean(stats['area_ratios'])
            area_std = np.std(stats['area_ratios'])
            print(f"  Corner area ratio:      {area_mean:.4f} ± {area_std:.4f}")

    if stats['invalid_corner_reasons']:
        print(f"  Invalid reasons:")
        for reason, count in stats['invalid_corner_reasons'].items():
            print(f"    - {reason}: {count}")

    print(f"\n  Output directories:")
    print(f"    Images:   {OUTPUT_BASE / split_name / 'images'}")
    print(f"    Matrices: {OUTPUT_BASE / split_name / 'matrices'}")
    print(f"    Debug:    {OUTPUT_BASE / split_name / 'debug'}")

# Overall totals
total_imgs = sum(s['total_images'] for s in all_stats.values())
total_warped = sum(s['warped_ok'] for s in all_stats.values())
print(f"\n{'='*80}")
print(f"OVERALL TOTALS:")
print(f"  Total images processed: {total_imgs}")
print(f"  Total warped:           {total_warped}")
if total_imgs > 0:
    print(f"  Overall success rate:   {100.0 * total_warped / total_imgs:.1f}%")
else:
    print(f"  Overall success rate:   N/A")


BATCH WARPING SUMMARY (UNet-based)

TRAIN SPLIT:
  Total images:           129
  Warped successfully:    129
  Skipped (no mask):      0
  Skipped (no corners):   0
  Skipped (invalid):      0
  Success rate:           100.0%
  Corner area ratio:      0.6134 ± 0.0310

  Output directories:
    Images:   /Users/ibrahimcbc/Library/CloudStorage/GoogleDrive-icebecioglu21@ku.edu.tr/My Drive/COMP411/assignments/COMP-411_Project/chessvision/data/processed/warped_unet/train/images
    Matrices: /Users/ibrahimcbc/Library/CloudStorage/GoogleDrive-icebecioglu21@ku.edu.tr/My Drive/COMP411/assignments/COMP-411_Project/chessvision/data/processed/warped_unet/train/matrices
    Debug:    /Users/ibrahimcbc/Library/CloudStorage/GoogleDrive-icebecioglu21@ku.edu.tr/My Drive/COMP411/assignments/COMP-411_Project/chessvision/data/processed/warped_unet/train/debug

VALIDATION SPLIT:
  Total images:           42
  Warped successfully:    42
  Skipped (no mask):      0
  Skipped (no corners):   0
  Skipped (in

## F) Sanity Check: Homography Matrix Verification

In [ ]:
# Sanity check: verify homography matrices are correct
print("="*80)
print("SANITY CHECK: Homography Matrix Verification")
print("="*80)

# Pick 3 random warped images for verification
test_samples = []
for split_name in ['train', 'validation', 'test']:
    images_dir = OUTPUT_BASE / split_name / "images"
    matrices_dir = OUTPUT_BASE / split_name / "matrices"

    if not images_dir.exists():
        continue

    warped_files = list(images_dir.glob("*.png"))[:1]  # Take first from each split
    for warped_file in warped_files:
        stem = warped_file.stem
        M_path = matrices_dir / f"{stem}_M.npy"
        Minv_path = matrices_dir / f"{stem}_Minv.npy"

        if M_path.exists() and Minv_path.exists():
            test_samples.append((split_name, stem, M_path, Minv_path))

if len(test_samples) == 0:
    print("No warped images found for sanity check")
else:
    print(f"\nTesting {len(test_samples)} sample(s)...\n")

    max_errors = []
    for split_name, stem, M_path, Minv_path in test_samples[:3]:
        M = np.load(str(M_path))
        Minv = np.load(str(Minv_path))

        # Test: warped corners -> original -> warped (round-trip)
        warped_corners = np.array([
            [0, 0],
            [WARP_SIZE, 0],
            [WARP_SIZE, WARP_SIZE],
            [0, WARP_SIZE]
        ], dtype=np.float32).reshape(-1, 1, 2)

        # Transform back to original
        original_corners = cv2.perspectiveTransform(warped_corners, Minv)

        # Transform forward again
        warped_corners_reprojected = cv2.perspectiveTransform(original_corners, M)

        # Compute error
        errors = np.linalg.norm(warped_corners - warped_corners_reprojected, axis=2)
        max_error = np.max(errors)
        max_errors.append(max_error)

        print(f"  {split_name}/{stem}:")
        print(f"    Max reprojection error: {max_error:.4f} pixels")

    overall_max_error = max(max_errors) if max_errors else 0.0
    threshold = 0.1  # Very small threshold for 800x800 images

    print(f"\n  Overall max error: {overall_max_error:.4f} pixels")
    if overall_max_error < threshold:
        print(f"  ✓ PASS: All reprojection errors < {threshold} pixels")
    else:
        print(f"  ✗ FAIL: Some errors >= {threshold} pixels")

SANITY CHECK: Homography Matrix Verification

Testing 3 sample(s)...

  train/3bab0eaaeb63a2ac9ae4942df4006a25_jpg.rf.b78947d5207c15119ee81058a1b75c1e:
    Max reprojection error: 0.0000 pixels
  validation/6f0de9b594de9f9b92c6a20daa51a28a_jpg.rf.d10980ee6f2a7a600d16ff48d980a15c:
    Max reprojection error: 0.0000 pixels
  test/8ff64b3f770bfe96bdffc629efd16460_jpg.rf.7b4792b9f562b28d55342586be82fe91:
    Max reprojection error: 0.0000 pixels

  Overall max error: 0.0000 pixels
  ✓ PASS: All reprojection errors < 0.1 pixels
